# Notebook 7, Part 3: Social Network Analysis

**Atlas source reading copy · local-server**

[Course guide](index.ipynb) · [Pinned original](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/NB07-MDR-TB-Case-Study-Part-3.ipynb)



Use the source local/server environment for native GIS, shell, installed project or legacy dependencies; Lab provides a reading/editing copy. Follow notebook order to regenerate omitted outputs; optional unlicensed media are external continuations.

**Provenance:** `NB07-MDR-TB-Case-Study-Part-3.ipynb` at `6d50f558cd39b40089cd3eea1fbc19425e5191d1`; original SHA-256 `3d3f2f2f768a3999f6d0d0bd73cdbb1b6c7acffeda1e07d5ccade7e5ead707cc`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


In [ ]:
%run ./resources/library.py

In [ ]:
style_notebook()

Digital Case Study: Multidrug-Resistant Tuberculosis (MDR-TB) Outbreak - Revisiting the 2005 Outbreak Investigation in Thailand by John Oeltmann

# Notebook 7, Part 3: Social Network Analysis

## Set up environment, load data

In [ ]:
import pandas as pd

pd.__version__

In [ ]:
pd.set_option('display.max_rows', None)  
pd.set_option('display.max_columns', None)  
pd.set_option('max_colwidth', None)  
pd.set_option('display.width', 1000)

Let's load the two batches of data for nodes and edges.

In [ ]:
nodes_df5 = pd.read_pickle('outputs/nodes_df5.pickle') # nodes / cases (MDR-TB only)
edges_df6 = pd.read_pickle('outputs/edges_df6.pickle') # edges / links

In [ ]:
nodes_df5.info()

In [ ]:
nodes_df5.head()

## Using `bokeh` and `networkx` for Network Visualization

From Wikipedia, on [bokeh](https://en.wikipedia.org/wiki/Bokeh): 
> In photography, bokeh (/ˈboʊkeɪ/ BOH-kay; Japanese: [boke]) is the aesthetic quality of the blur produced in the out-of-focus parts of an image produced by a lens. Bokeh has been defined as "the way the lens renders out-of-focus points of light".

From the Python bokeh [website](https://bokeh.pydata.org/en/latest/):
> Bokeh is an interactive visualization library that targets modern web browsers for presentation. Its goal is to provide elegant, concise construction of versatile graphics, and to extend this capability with high-performance interactivity over very large or streaming datasets. Bokeh can help anyone who would like to quickly and easily create interactive plots, dashboards, and data applications.

On [NetworkX](https://networkx.github.io/):
> NetworkX is a Python package for the creation, manipulation, and study of the structure, dynamics, and functions of complex networks.

## Graphs, nodes and edges

A graph is made of nodes and edges. 

Let's set these up using our loaded dataframes.

In [ ]:
nodes = list(nodes_df5['CaseNo'])

nodes

In [ ]:
edges = [ tuple(x) for x in edges_df6[['CaseNo1', 'CaseNo2']].values ]

edges

Let's set up the attributes for our node interactive pop-ups.

In [ ]:
attributes = {}
for each in nodes_df5.iterrows():
    new_dict = {each[1]['CaseNo']: {'CaseNo': each[1]['CaseNo'],\
                              'DrugResType': each[1]['DRTYPE'],\
                              'Links': each[1]['links']}}
    attributes.update(new_dict)

attributes

... and the tooltips where these will appear.

In [ ]:
tooltips = [
    ("Case No:", "@CaseNo"),
    ("Drug Res Type:", "@DrugResType"),
    ("Links:", "@Links")
]

Let's run our `bokeh` and `networkx` Python code.

In [ ]:
import networkx as nx

from bokeh.io import show, output_notebook
from bokeh.models import Plot, Range1d, HoverTool, TapTool, BoxSelectTool, StaticLayoutProvider, Circle, MultiLine
from bokeh.models.graphs import NodesAndLinkedEdges, EdgesAndLinkedNodes
from bokeh.palettes import Spectral4
from bokeh.layouts import row
from bokeh.plotting import from_networkx


In [ ]:
output_notebook()

## Visualizing networks

### Setting up `G` as a `networkx` graph and adding nodes and edges

In [ ]:
G = nx.Graph()
G.add_nodes_from(nodes)
G.add_edges_from(edges)

nx.set_node_attributes(G, attributes)

### Visualizing using NetworkX (quick check)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

pos1 = nx.circular_layout(G)
pos2 = nx.spring_layout(G)

plt.figure(1, figsize=(5,5))
plt.title='Test'
nx.draw(G, pos1)

plt.figure(2, figsize=(5,5))
nx.draw(G,pos2) 

plt.show()


### Interactive visualization using Bokeh

#### Spring layout

In [ ]:

# Assuming G is your graph and tooltips are predefined
# 1. Relabel nodes to integers (if needed)
G = nx.relabel_nodes(G, {node: i for i, node in enumerate(G.nodes())})

# 2. Compute the layout
layout = nx.spring_layout(G, scale=1, center=(0, 0))

# Convert numpy arrays to lists (Bokeh requires JSON-serializable values)
layout = {k: list(v) for k, v in layout.items()}

# 3. Set up the plot
plot1 = Plot(width=450, height=450,
             x_range=Range1d(-1.1, 1.1), y_range=Range1d(-1.1, 1.1))
plot1.title.text = "MDR-TB Links - Spring Layout"
plot1.add_tools(HoverTool(tooltips=tooltips), TapTool(), BoxSelectTool())

# 4. Use the layout with from_networkx
graph_renderer1 = from_networkx(G, layout)

# 5. Set node renderer
graph_renderer1.node_renderer.glyph = Circle(size=20, fill_color=Spectral4[0])
graph_renderer1.node_renderer.selection_glyph = Circle(size=20, fill_color=Spectral4[2])
graph_renderer1.node_renderer.hover_glyph = Circle(size=20, fill_color=Spectral4[1])

graph_renderer1.edge_renderer.glyph = MultiLine(line_color="#CCCCCC", line_alpha=0.8, line_width=2)
graph_renderer1.edge_renderer.selection_glyph = MultiLine(line_color=Spectral4[2], line_width=5)
graph_renderer1.edge_renderer.hover_glyph = MultiLine(line_color=Spectral4[1], line_width=5)

# 6. Set selection/inspection policies
graph_renderer1.selection_policy = NodesAndLinkedEdges()
graph_renderer1.inspection_policy = NodesAndLinkedEdges()

# 7. Add renderer to the plot
plot1.renderers.append(graph_renderer1)

#### Circular layout

In [ ]:
# circular layout

# circular layout
plot2 = Plot(width=450, height=450,  # Change plot_width and plot_height to width and height
             x_range=Range1d(-1.1, 1.1), y_range=Range1d(-1.1, 1.1))
plot2.title.text = "MDR-TB Links - Circular Layout"
plot2.add_tools(HoverTool(tooltips=tooltips), TapTool(), BoxSelectTool())

# Create graph renderer using circular layout
graph_renderer2 = from_networkx(G, nx.circular_layout, scale=1, center=(0, 0))

# Set visual styles for nodes and edges
graph_renderer2.node_renderer.glyph = Circle(size=20, fill_color=Spectral4[0])
graph_renderer2.node_renderer.selection_glyph = Circle(size=20, fill_color=Spectral4[2])
graph_renderer2.node_renderer.hover_glyph = Circle(size=20, fill_color=Spectral4[1])

graph_renderer2.edge_renderer.glyph = MultiLine(line_color="#CCCCCC", line_alpha=0.8, line_width=2)
graph_renderer2.edge_renderer.selection_glyph = MultiLine(line_color=Spectral4[2], line_width=5)
graph_renderer2.edge_renderer.hover_glyph = MultiLine(line_color=Spectral4[1], line_width=5)

# Set selection/inspection policies
graph_renderer2.selection_policy = NodesAndLinkedEdges()
graph_renderer2.inspection_policy = NodesAndLinkedEdges()

# Add renderer to the plot
plot2.renderers.append(graph_renderer2)

#### Showing spring and circular layouts side by  side

In [ ]:
show(row(plot1, plot2))

### Underlying data

In [ ]:
graph_renderer1.node_renderer.data_source.data

### Graph information

In [ ]:
G.number_of_nodes()

In [ ]:
G.number_of_edges()

## Case Study Questions

1. What information can you glean from the two network displays?

Type answer here.

## References

1. Kapoor A. Social Network Analysis in Python. URL: https://www.datacamp.com/community/tutorials/social-network-analysis-python (last accessed 7/21/2019)